# Submit Vertex AI TPU Training from GKE

This notebook demonstrates submitting a Vertex AI Custom Job with TPU accelerators from a JupyterHub instance running on GKE. Authentication uses Workload Identity — no JSON key files needed.

**What this notebook does:**
1. Verifies Workload Identity authentication
2. Writes a Dockerfile and training script (CIFAR-10 on TPU)
3. Builds and pushes the container image via Cloud Build
4. Submits a Vertex AI Custom Job on TPU v5e
5. Optionally runs the same job as a Vertex Pipeline

**Google Cloud services used:** Vertex AI, Artifact Registry, Cloud Build, Cloud Storage, GKE

## 1. Install Packages

Install the following packages required to execute this notebook.

In [ ]:
! pip3 install --upgrade --quiet google-cloud-aiplatform \
                                 google-cloud-storage \
                                 kfp \
                                 google-cloud-pipeline-components

### Check tool versions

In [ ]:
! kubectl version --client
! python3 -c "import kfp; print('KFP SDK version: {}'.format(kfp.__version__))"
! python3 -c "import google_cloud_pipeline_components; print('google_cloud_pipeline_components version: {}'.format(google_cloud_pipeline_components.__version__))"

## Verify GKE Environment and Workload Identity

In [ ]:
import subprocess

result = subprocess.run(
    ["cat", "/var/run/secrets/kubernetes.io/serviceaccount/namespace"],
    capture_output=True, text=True,
)
print(f"Running in Kubernetes namespace: {result.stdout}")

from google.auth import default

credentials, project = default()
print(f"Authenticated to project: {project}")
print("Workload Identity is working - no JSON key files needed")

## Set Project Parameters

Set up required parameters.

- `PROJECT_ID`: Google Cloud project ID
- `REGION`: Google Cloud region (e.g. us-central1)

In [ ]:
# Project parameters
PROJECT_ID = "sandbox-401718"          # @param {type:"string"}
REGION = "us-central1"                  # @param {type:"string"}

## TPU Training Image Preparation

Set up required parameters.

Container Image:
- `VERSION`: Version or tag of the Docker image. Default set as `latest`
- `REPO_NAME`: The name of the Artifact Registry repository that will store the container image
- `TRAIN_IMAGE_ID`: The name of the image that will be used to run TPU training. The full image name: `<REGION>-docker.pkg.dev/<PROJECT_ID>/<REPO_NAME>/<TRAIN_IMAGE_ID>:<VERSION>`

Custom Job and Pipeline:
- `SERVICE_ACCOUNT`: The service account to use to run custom jobs and pipeline
- `PIPELINE_ROOT`: Cloud Storage bucket URI for pipeline artifacts
- `PIPELINE_REPO`: The name of the Artifact Registry repository that will store the compiled pipeline file
- `PIPELINE_NAME`: The name of the Vertex Pipeline that is run

In [ ]:
# Image Parameters
VERSION = "latest"
REPO_NAME = "tpu-training-repository"    # @param {type:"string"}
TRAIN_IMAGE_ID = "tpu-train"             # @param {type:"string"}

# Vertex Custom Job parameters
SERVICE_ACCOUNT="757654702990-compute@developer.gserviceaccount.com" # @param {type:"string"}
PIPELINE_ROOT = f"gs://{PROJECT_ID}-tpu-pipeline-root"     # @param {type:"string"}
PIPELINE_REPO = "tpu-training-kfp"       # @param {type:"string"}
PIPELINE_NAME = "tpu-training-pipeline"  # @param {type:"string"}

# Working dir for model artifacts
WORKING_DIR = f"{PIPELINE_ROOT}/model"

### Write the Dockerfile

Define the instructions to set the TPU environment and run the training script in a Dockerfile.

The container uses the Vertex AI prebuilt TPU base image, installs the TensorFlow TPU wheel and `libtpu` shared library, and copies the training script.

Learn more about [custom container training with TPU accelerators](https://cloud.google.com/vertex-ai/docs/training/training-with-tpu-vm#custom_container).

In [ ]:
! mkdir -p tpu-container-artifacts

In [ ]:
%%writefile tpu-container-artifacts/Dockerfile

# Specifies base image and tag
FROM us-docker.pkg.dev/vertex-ai/training/tf-tpu-pod-base-cp310:latest
WORKDIR /root

# Download and install `tensorflow` (2.16.1 for V5e support).
RUN pip install https://storage.googleapis.com/cloud-tpu-tpuvm-artifacts/tensorflow/tf-2.16.1/tensorflow-2.16.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

# Download and install `libtpu` (1.10.0 for V5e support).
# You must save `libtpu.so` in the '/lib' directory of the container image.
RUN curl -L https://storage.googleapis.com/cloud-tpu-tpuvm-artifacts/libtpu/1.10.0/libtpu.so -o /lib/libtpu.so

# Download and install tensorflow-datasets
RUN pip3 install tensorflow-datasets

# Copies the trainer code to the docker image.
COPY train.py /root/train.py

# Sets the default command to run train.py.
CMD ["python3", "train.py"]

### Create the Training Script

Write the contents of the training script to *train.py*.

In summary, the training script does the following:

- Gets the directory where to save the model artifacts from the environment variable `AIP_MODEL_DIR`. This variable is set by the training service.
- Loads CIFAR10 dataset from TF Datasets (tfds).
- Builds a model using TF.Keras model API.
- Compiles the model.
- Sets a training distribution strategy according to the argument `args.distribute`.
- Trains the model with epochs and steps according to the arguments `args.epochs` and `args.steps`
- Saves the trained model to the specified model directory.
- Runs TPU specific tasks:
    - Finds the TPU cluster, connects to the cluster, and sets the training strategy to TPUStrategy.
    - Saves the trained TPU model to the local device so that the model can be saved to the location in `AIP_MODEL_DIR`.

In [ ]:
%%writefile tpu-container-artifacts/train.py
# Single, Mirror and Multi-Machine Distributed Training for CIFAR-10

import tensorflow_datasets as tfds
import tensorflow as tf
from tensorflow.python.client import device_lib
import argparse
import os
import sys
tfds.disable_progress_bar()

parser = argparse.ArgumentParser()
parser.add_argument('--lr', dest='lr',
                    default=0.01, type=float,
                    help='Learning rate.')
parser.add_argument('--epochs', dest='epochs',
                    default=10, type=int,
                    help='Number of epochs.')
parser.add_argument('--steps', dest='steps',
                    default=200, type=int,
                    help='Number of steps per epoch.')
parser.add_argument('--distribute', dest='distribute', type=str, default='single',
                    help='distributed training strategy')
args = parser.parse_args()

print('Python Version = {}'.format(sys.version))
print('TensorFlow Version = {}'.format(tf.__version__))
print('TF_CONFIG = {}'.format(os.environ.get('TF_CONFIG', 'Not found')))
print('DEVICES', device_lib.list_local_devices())

# Single Machine, single compute device
if args.distribute == 'single':
    if tf.test.is_gpu_available():
        strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
    else:
        strategy = tf.distribute.OneDeviceStrategy(device="/cpu:0")
# Single Machine, multiple TPU devices
elif args.distribute == 'tpu':
    cluster_resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu="local")
    tf.config.experimental_connect_to_cluster(cluster_resolver)
    tf.tpu.experimental.initialize_tpu_system(cluster_resolver)
    strategy = tf.distribute.TPUStrategy(cluster_resolver)
    print("All devices: ", tf.config.list_logical_devices('TPU'))
# Single Machine, multiple compute device
elif args.distribute == 'mirror':
    strategy = tf.distribute.MirroredStrategy()
# Multiple Machine, multiple compute device
elif args.distribute == 'multi':
    strategy = tf.distribute.experimental.MultiWorkerMirroredStrategy()

# Multi-worker configuration
print('num_replicas_in_sync = {}'.format(strategy.num_replicas_in_sync))

# Preparing dataset
BUFFER_SIZE = 10000
BATCH_SIZE = 64

def make_datasets_unbatched():
  # Scaling CIFAR10 data from (0, 255] to (0., 1.]
  def scale(image, label):
    image = tf.cast(image, tf.float32)
    image /= 255.0
    return image, label

  datasets, info = tfds.load(name='cifar10',
                            with_info=True,
                            as_supervised=True)
  return datasets['train'].map(scale).cache().shuffle(BUFFER_SIZE).repeat()


# Build the Keras model
def build_and_compile_cnn_model():
  model = tf.keras.Sequential([
      tf.keras.layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)),
      tf.keras.layers.MaxPooling2D(),
      tf.keras.layers.Conv2D(32, 3, activation='relu'),
      tf.keras.layers.MaxPooling2D(),
      tf.keras.layers.Flatten(),
      tf.keras.layers.Dense(10, activation='softmax')
  ])
  model.compile(
      loss=tf.keras.losses.sparse_categorical_crossentropy,
      optimizer=tf.keras.optimizers.SGD(learning_rate=args.lr),
      metrics=['accuracy'])
  return model

# Train the model
NUM_WORKERS = strategy.num_replicas_in_sync
# Here the batch size scales up by number of workers since
# `tf.data.Dataset.batch` expects the global batch size.
GLOBAL_BATCH_SIZE = BATCH_SIZE * NUM_WORKERS
MODEL_DIR = os.getenv("AIP_MODEL_DIR")

train_dataset = make_datasets_unbatched().batch(GLOBAL_BATCH_SIZE)

with strategy.scope():
  # Creation of dataset, and model building/compiling need to be within
  # `strategy.scope()`.
  model = build_and_compile_cnn_model()

model.fit(x=train_dataset, epochs=args.epochs, steps_per_epoch=args.steps)
if args.distribute=="tpu":
    save_locally = tf.saved_model.SaveOptions(experimental_io_device='/job:localhost')
    model.export(MODEL_DIR, options=save_locally)
else:
    model.export(MODEL_DIR)

### Build the Training Container Image

Build and push the training container image to Artifact Registry.

1. Enable the Artifact Registry API.
1. Create a private repository in Artifact Registry.
1. Configure authentication to Artifact Registry.
1. Submit the training container image using Cloud Build.

In [ ]:
# Set the project id
! gcloud config set project {PROJECT_ID}
# Enable Artifact Registry
! gcloud services enable artifactregistry.googleapis.com

In [ ]:
# Create the repository
! gcloud artifacts repositories create $REPO_NAME --repository-format=docker \
    --location=$REGION --description="Vertex TPU training repository" || true

In [ ]:
! gcloud auth configure-docker $REGION-docker.pkg.dev --quiet

In [ ]:
# Set the training container image
TRAIN_IMAGE = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/{REPO_NAME}/{TRAIN_IMAGE_ID}:{VERSION}"

In [ ]:
# Submit the build source
! gcloud builds submit tpu-container-artifacts --region={REGION} --tag={TRAIN_IMAGE}

## Custom Jobs

Submit TPU training workloads using [Custom Jobs and Worker Pool Specs](https://cloud.google.com/vertex-ai/docs/training/create-custom-job).

In this example, a CIFAR-10 classification model is trained using TPU accelerators.

Learn more about [Training with TPU accelerators](https://cloud.google.com/vertex-ai/docs/training/training-with-tpu-vm).

In [ ]:
# Import libraries

import os
import kfp
from kfp.registry import RegistryClient
from kfp import compiler
import kfp.components as comp
from google.cloud import aiplatform
from google.cloud.aiplatform import gapic
from google_cloud_pipeline_components.v1.custom_job import CustomTrainingJobOp

### Set Hardware Accelerators and Training Parameters

Set the TPU accelerator type and count for training. See the [locations where accelerators are available](https://cloud.google.com/vertex-ai/docs/general/locations#accelerators).

In [ ]:
[x for x in dir(gapic.AcceleratorType) if 'TPU' in x]

In [ ]:
TRAIN_COMPUTE = "ct5lp-hightpu-1t"
TPU_TOPOLOGY = "1x1"
TRAIN_STRATEGY = "single"

EPOCHS = 20
STEPS = 10000

TRAINER_ARGS = [
  "--epochs=" + str(EPOCHS),
  "--steps=" + str(STEPS),
  "--distribute=" + TRAIN_STRATEGY,
]

print("Train machine type:", TRAIN_COMPUTE)
print("TPU Topology:", TPU_TOPOLOGY)
print("Trainer args:", TRAINER_ARGS)

In [ ]:
# WORKER_POOL_SPEC = [
#     {
#         "replica_count": 1,
#         "machine_spec": {
#             "machine_type": TRAIN_COMPUTE,
#             "tpu_topology": TPU_TOPOLOGY,
#         },
#         "container_spec": {
#             "image_uri": TRAIN_IMAGE,
#             "args": TRAINER_ARGS,
#             "env": [
#                 {"name": "AIP_MODEL_DIR", "value": WORKING_DIR},
#                 # Environment variables for 1x1 topology
#                 {"name": "TPU_CHIPS_PER_HOST_BOUNDS", "value": "1,1,1"},
#                 {"name": "TPU_HOST_BOUNDS", "value": "1,1,1"},
#             ],
#         },
#     }
# ]

WORKER_POOL_SPEC = [
  {
    "replica_count": 1,
    "machine_spec": {
      "machine_type": TRAIN_COMPUTE,
      "tpu_topology": TPU_TOPOLOGY,
    },
    "container_spec": {
      "image_uri": TRAIN_IMAGE,
      "command": ["python3", "train.py"],  # <--- Add this line
      "args": TRAINER_ARGS,
      "env": [
        {"name": "AIP_MODEL_DIR", "value": WORKING_DIR},
        # Environment variables for 1x1 topology
        {"name": "TPU_CHIPS_PER_HOST_BOUNDS", "value": "1,1,1"},
        {"name": "TPU_HOST_BOUNDS", "value": "1,1,1"},
      ],
    },
  }
]

In [ ]:
aiplatform.init(project=PROJECT_ID, location=REGION, staging_bucket=PIPELINE_ROOT)

custom_job = aiplatform.CustomJob(
    display_name="gke-submitted-tpu-training-job",
    worker_pool_specs=WORKER_POOL_SPEC,
    project=PROJECT_ID,
    location=REGION,
    staging_bucket=PIPELINE_ROOT,
)

custom_job.run(sync=False, service_account=SERVICE_ACCOUNT)

In [ ]:
custom_job

## Run Custom Job Component in a Kubeflow Pipeline

Users have the option to incorporate the custom job into a Vertex Pipeline as part of a larger training and modeling work flow. Pipeline is stored and versioned in Artifact Registry.

In [ ]:
# Create the KFP Artifact Registry repo
! gcloud artifacts repositories create {PIPELINE_REPO} --repository-format=kfp \
    --location={REGION} --description="TPU training pipelines" || true

@kfp.dsl.pipeline(pipeline_root=PIPELINE_ROOT, name=PIPELINE_NAME)
def pipeline():
    tpu_job = CustomTrainingJobOp(
        project=PROJECT_ID,
        location=REGION,
        display_name="tpu-training-workload",
        worker_pool_specs=WORKER_POOL_SPEC,
        service_account=SERVICE_ACCOUNT,
    ).set_display_name("tpu-training-workload")


# Compile Pipeline Job
compiler.Compiler().compile(
    pipeline_func=pipeline, package_path=os.path.join("/tmp", "tpu_training_pipeline.yaml")
)

# Upload Pipeline to Artifact Registry
client = RegistryClient(
    host=f"https://{REGION}-kfp.pkg.dev/{PROJECT_ID}/{PIPELINE_REPO}"
)
templateName, versionName = client.upload_pipeline(
    file_name=os.path.join("/tmp", "tpu_training_pipeline.yaml"), tags=["v1", VERSION]
)

# Configure the pipeline
job = aiplatform.PipelineJob(
    display_name="tpu_training_pipeline",
    template_path=f"https://{REGION}-kfp.pkg.dev/{PROJECT_ID}/{PIPELINE_REPO}/{PIPELINE_NAME}/{VERSION}",
    pipeline_root=PIPELINE_ROOT,
    enable_caching=False,
)

# Run the job
job.run(sync=False)

## Check Task Completion and Output

Once the job and/or pipeline is complete, check to see the output of the TPU training job.

In [ ]:
# Check the status of the custom job
custom_job.state

## Cleanup

To tear down all infrastructure (GKE cluster, IAM, JupyterHub), run `terraform destroy` from the `infra/` directory.

In [ ]:
# Delete the Artifact Registry repository
! gcloud artifacts repositories delete $REPO_NAME --location=$REGION --quiet

# Remove the training container artifacts
! rm -rf tpu-container-artifacts

## Additional References

* [Training with TPU accelerators](https://cloud.google.com/vertex-ai/docs/training/training-with-tpu-vm)
* [Custom container training with TPU](https://cloud.google.com/vertex-ai/docs/training/training-with-tpu-vm#custom_container)
* [TPU types](https://cloud.google.com/tpu/docs/system-architecture-tpu-vm)
* [Prebuilt containers for training](https://cloud.google.com/vertex-ai/docs/training/pre-built-containers)
* [Vertex AI Pipelines](https://cloud.google.com/vertex-ai/docs/pipelines)
* [GKE Workload Identity](https://cloud.google.com/kubernetes-engine/docs/how-to/workload-identity)